# Lecture 22 - Shaping the build

**Lecture Topic**: How software projects are run from start to finish: the development lifecycle, deciding what to build and when it is done, and what happens afterwards, when a project has to be maintained (or allowed to end).

## Announcements

- **M3 (final repository and report) is due Friday Dec 4th.**
- **Presentations** are Nov 30, Dec 2, and Dec 4. Team interviews are booked that week too.
- **Next session (Mon Nov 23) includes the quiz** (about 20 minutes, closed-AI, 10%) on the material since the midterm. It replaces the final exam. Then the (remote) clinic on Wednesday.
- ClassBuzz poll for today: `LEZA`

## Where we are

This is the last stretch, Module E, and it steps back from the code.

 - **A**: the data tools. **B**: working with an agent. **C**: predicting scale. **D**: building it well.
 - **E**: how a project is run, from deciding what to build to keeping it alive afterwards.

Today:
 - The software development lifecycle: waterfall, the V-model, and agile
 - Linking the tests back to the spec
 - Deciding what to build, and what "done" means
 - Maintenance, open source, and when to let a project die

# The software development lifecycle

Every software project, whether it is a class project or a banking system, goes through the same stages. Different methods arrange them differently, but the stages are always there:

| Stage | The question | Where we met it |
|---|---|---|
| **Requirements** | What should it do, and for whom? | the spec (Module B, M1) |
| **Design** | How will it be organised? | architecture and structure (Lectures 19-20) |
| **Implementation** | Write it | you, or more and more, the agent |
| **Testing** | Does it do what the requirements said? | Lecture 21 |
| **Deployment** | Get it to the people who use it | CI, packaging, a dashboard |
| **Maintenance** | Keep it working as the world changes | today |

The interesting question is how you move through them: once, in order, or round and round.

## Waterfall: one stage at a time

The **waterfall** model does each stage once, in order, and finishes each before starting the next. Requirements are written and signed off, then the design, then the code, then testing at the end.

It is usually traced to a 1970 paper by Winston Royce. The funny thing is that Royce drew the one-pass diagram as an example of what *not* to do. He wrote that the approach "is risky and invites failure", because problems are only found in testing, at the very end, when they are most expensive to fix. He argued for doing it twice and getting feedback from the customer throughout. Most people only copied the diagram.

Waterfall is still the right fit when requirements really are fixed and mistakes are very expensive: safety-critical systems, hardware, medical devices, work done to a contract. For most data science, requirements change as soon as someone sees the first result, so finding out at the end is too late.

## The V-model: every test traces back to a piece of the spec

A useful refinement of waterfall is the **V-model**. Fold the waterfall in half, so that every level of the spec on the way down has a matching level of testing on the way up:

```
 Requirements  ───────────────────────────  Acceptance tests
     "what the user needs"                      "does it solve their problem?"
   System design  ─────────────────────  System / integration tests
       "how the pieces fit"                    "do the pieces work together?"
     Module design  ───────────────  Unit tests
         "what each function promises"     "does each function keep its promise?"
                    Implementation
```

This is the idea to keep, whatever method you use: **every test should trace back to something in the spec, and everything in the spec should have a test.** A requirement with no test is a hope. A test with no requirement behind it is checking something nobody asked for.

In Lecture 21 the unit tests checked `tip_rate`'s promises, the golden test checked the whole report (a system test), and the tests we wrote for the agent's PR came from the task description, "the highest average tips", which makes them acceptance tests.

## Agile: short loops and working software

By the 1990s many teams found that big up-front plans kept going wrong, and in 2001 a group of them wrote the **Agile Manifesto**. Its four values are:

 - **Individuals and interactions** over processes and tools
 - **Working software** over comprehensive documentation
 - **Customer collaboration** over contract negotiation
 - **Responding to change** over following a plan

(They are careful to say the things on the right still have value, just less than the things on the left.)

In practice agile means going round the whole lifecycle in short loops, usually one to four weeks, called sprints or iterations. Each loop ends with something working that a user can look at, and what you learn feeds into the next loop. **Scrum** is the most common recipe (a backlog of work, sprints, a short daily stand-up, a review at the end of each sprint). **Kanban** drops the sprints and just limits how much work is in progress at once.

## User stories and acceptance criteria

Agile teams write requirements as short **user stories**, each with **acceptance criteria**:

> **As a** city transport analyst, **I want** the typical tip percentage for each pickup zone, **so that** I can find zones where drivers are under-tipped.
>
> **Acceptance criteria**
> 1. Every zone with at least 20 card trips appears, once.
> 2. Tips are a percentage of the fare, between 0 and 100.
> 3. Cash trips are left out, since their tips are not recorded.
> 4. The result is sorted with the lowest-tipping zones first.

The story says who it is for and why, which is what stops you building the wrong thing. The acceptance criteria say when it is done. And each criterion can be written straight down as a test, which is the V-model's top level.

Going one step further, **test-driven development** (TDD) writes the test *before* the code: red (a failing test), green (just enough code to pass), then refactor. Behaviour-driven development writes the criteria in a Given/When/Then form that is close to plain English.

In [ ]:
# set-up for the demo: a fresh folder for the files, and make sure `!pytest` uses this environment
import os, shutil, sys
from pathlib import Path
shutil.rmtree("build_demo", ignore_errors=True)
Path("build_demo").mkdir()
os.environ["PATH"] = str(Path(sys.executable).parent) + os.pathsep + os.environ["PATH"]

Here are those four acceptance criteria, written as tests before any code exists:

In [ ]:
%%writefile build_demo/test_story.py
from pathlib import Path
import pandas as pd
import pytest
from tip_report import zone_tip_pct

DATA = Path(__file__).parents[2] / "data" / "2019_Yellow_Taxi_Trip_Data.csv"

@pytest.fixture(scope="module")
def trips():
    return pd.read_csv(DATA)

def test_every_busy_zone_once(trips):                        # criterion 1
    card = trips[trips["payment_type"] == 1]
    busy = card["pulocationid"].value_counts()
    report = zone_tip_pct(trips)
    assert set(report.index) == set(busy[busy >= 20].index)
    assert report.index.is_unique

def test_tips_are_percentages(trips):                        # criterion 2
    assert zone_tip_pct(trips).between(0, 100).all()

def test_cash_trips_left_out():                              # criterion 3
    small = pd.DataFrame({"pulocationid": [1] * 21, "payment_type": [1] * 20 + [2],
                          "fare_amount": [10.0] * 21, "tip_amount": [2.0] * 20 + [0.0]})
    assert zone_tip_pct(small).loc[1] == pytest.approx(20.0)

def test_lowest_first(trips):                                # criterion 4
    assert zone_tip_pct(trips).is_monotonic_increasing

With no code yet, every test fails, which is the "red" step. That is the spec, in a form the agent can run.

In [ ]:
%%writefile build_demo/tip_report.py
def zone_tip_pct(trips):
    raise NotImplementedError

In [ ]:
!cd build_demo && pytest -q --tb=no test_story.py

Now we (or the agent) write just enough code to pass:

In [ ]:
%%writefile build_demo/tip_report.py
def zone_tip_pct(trips, min_trips=20):
    """Median tip as a percentage of fare, per pickup zone, card trips only, lowest first."""
    card = trips[(trips["payment_type"] == 1) & (trips["fare_amount"] > 0)]
    pct = 100 * card["tip_amount"] / card["fare_amount"]
    by_zone = pct.groupby(card["pulocationid"])
    return by_zone.median()[by_zone.size() >= min_trips].clip(0, 100).sort_values()

In [ ]:
!cd build_demo && pytest -q test_story.py

Note, writing the code turned up a question the story did not answer: what to do with a fare of zero, and with the odd trip whose tip is bigger than its fare. We made a choice (drop zero fares, cap at 100%), and the right move is to go back and add it to the acceptance criteria, so the spec and the tests stay in step.

## What changes with agents

Agents make each loop of the lifecycle much faster. A sprint's worth of rough implementation can take an afternoon, and a prototype can take minutes. Two things follow:

 - **The spec and the tests matter more, not less.** The spec is what you give the agent, and the tests are how you check what comes back. This is the V-model again, run in a much tighter loop. Some teams now call this *spec-driven development*: write the spec and the acceptance tests first, then let the agent iterate until they pass.
 - **Waterfall-style mistakes come back if you are not careful.** Asking an agent to build a whole system from one big spec, then testing it at the end, is the waterfall: you find out what was wrong when it is most expensive to fix. Short loops, with a person looking at working output each time, still work better.

### Question 1

Your team's M2 spec says "the dashboard should load quickly". Which is the best way to make that testable?

 - A. Leave it as it is; everyone knows what quickly means
 - B. "The dashboard loads the full 2019 file and draws the main chart in under 3 seconds on DataHub", with a test that times it
 - C. Ask the agent to make it as fast as possible
 - D. Remove it from the spec

**B.** A requirement you cannot test is a hope. Putting a number on it, and a machine it is measured on, turns it into an acceptance criterion with a matching test (a scaling test, from last lecture). C sounds diligent but gives the agent no point at which it is done.

# Deciding what to build

For most of computing's history, the expensive step was building the thing. With an agent, a rough version of an idea is often an afternoon's work. So more of the hard work moves to the requirements end of the lifecycle: *what* to build, *for whom*, and *when it is good enough*. Those questions are about people and judgement, and the agent cannot answer them for you.

Real requests arrive vague: "can you make the taxi data useful?", "I want to understand tipping." Your first job is to find the **real need** behind the stated request. People often ask for a solution ("build me a dashboard") when what they have is a question ("which zones tip badly, and why?").

### Question 2

A researcher says: "I'd like a tool to see tipping across the city." What is the best first move?

 - A. Start building an interactive dashboard
 - B. Ask what question they are trying to answer, and what they will do with the result
 - C. Pick a charting library
 - D. Ask the agent to build "a tipping tool"

**B.** "A tool" is a guess at the solution. The real need might be answered by two charts in a notebook, or it might genuinely need a dashboard, and you cannot tell until you know the question and the decision it feeds. A few minutes of asking saves a day of building the wrong thing. The user story template ("as a ..., I want ..., so that ...") is a good way to force that conversation.

## Prototype to think

Because a rough version is cheap, you can build one to *learn* rather than to keep. A prototype is a question: does this approach work, does the user actually want this, is the data good enough? You build it, show it to someone, and often throw it away and improve the spec. This is the agile loop at its smallest.

The discipline is knowing when to stop prototyping and commit. Endless prototypes are their own kind of drift, and the acceptance criteria are what tell you the exploring is over.

## The trade-offs you are choosing between

Deciding what to build is choosing a point on four axes:

 - **Speed**: how fast do we need it?
 - **Cost**: compute, money, and your time to build and *maintain* it.
 - **Risk**: what is the damage if it is wrong, late, or breaks?
 - **Human effort**: how much ongoing attention will it need from a person?

A one-off answer for a meeting sits at one corner (fast, cheap, low stakes). A shared tool that people depend on sits at another (slower, costlier, higher stakes, and someone has to look after it). This is the three-shapes choice from the architecture lecture, now made before you write anything.

### Question 3

You need to answer one question for a decision being made this afternoon. The data is a bit messy. What do you build?

 - A. A clean, tested, documented package
 - B. A scheduled pipeline that refreshes nightly
 - C. A quick notebook that answers the question, with a note on the caveats
 - D. A dashboard so people can explore it later

**C.** For a one-off decision this afternoon, the quick notebook is the right amount, as long as you are honest about the caveats of the messy data. A, B and D all solve a problem nobody has yet, and each one is something you would then have to maintain. If people keep coming back to the question, *then* you build the durable version.

# Maintenance: most of a project's life

The lifecycle table makes maintenance look like one stage of six. In practice it is most of a project's life, and most estimates put it at well over half of the total cost of software. Code that nobody touches still breaks, because the world around it changes.

The usual way to split it up:

 - **Corrective**: fixing bugs people find.
 - **Adaptive**: keeping up with a changing world: new library versions, new data formats, a new operating system.
 - **Perfective**: improvements people ask for once they use it.
 - **Preventive**: tidying up so the next change is easier (the refactoring from Lecture 20).

## Adaptive maintenance happened to this course

The environment I use to present these slides is a good example.

 - **NumPy 2** removed `np.NaN` (it is `np.nan` now), so old code that used it stopped working. Several of last year's examples had to be fixed.
 - **pandas 3** changed strings to a new `str` dtype and turned on copy-on-write, which changes when modifying a DataFrame affects another one.
 - The slideshow tool, RISE, was written for the classic Jupyter notebook. When Jupyter moved on, the classic version stopped being developed, so my `environment.yml` pins `notebook=6.5.7`, `rise=5.7.1` and `setuptools<81`, with a comment explaining that these pins are load-bearing.

None of the course code had a bug. The world moved, and the code had to keep up. Pinning versions buys time, but every pin is a small debt: eventually the pinned versions stop installing, or have a security hole, and you have to move.

## Deprecate before you break

If other people use your code, change it in a way that gives them time to adapt. The standard pattern is to **deprecate** first: keep the old behaviour working for a while, but warn when it is used, and only remove it in a later version.

In [ ]:
import warnings

def tip_rate(fare, tip, percent=None):
    if percent is not None:
        warnings.warn("`percent` is deprecated and will be removed in version 2.0; "
                      "multiply the result by 100 instead", FutureWarning, stacklevel=2)
        if percent:
            return 100 * tip / fare
    return tip / fare

print(tip_rate(10, 2))
print(tip_rate(10, 2, percent=True))      # still works, but warns

This is what pandas and NumPy do, which is why you have seen so many `FutureWarning`s this quarter: they are the libraries giving you time before something changes. Together with **semantic versioning** (`MAJOR.MINOR.PATCH`, where only a major version is allowed to break things) and a changelog, it lets people upgrade on their own schedule.

## Maintaining open source

Almost everything we have used this quarter (NumPy, pandas, xarray, matplotlib, pytest, Jupyter) is open source, and much of it is maintained by a small number of people, many of them volunteers or paid for only part of their time.

<img src="images/xkcd_dependency.png" style="height: 380px">

<small>xkcd 2347, "Dependency", by Randall Munroe (<a href="https://xkcd.com/2347/">xkcd.com/2347</a>, CC BY-NC 2.5)</small>

It is a joke, but only just:

 - **2014, Heartbleed**: a serious bug in OpenSSL, which secures much of the web, was found in a project with very few core developers and very little funding.
 - **2016, left-pad**: a developer removed an 11-line package from npm, and builds broke across the JavaScript world, because thousands of projects depended on it without knowing.
 - **2024, xz utils**: an attacker spent about two years gaining the trust of the sole, overworked maintainer of a compression library, then slipped a backdoor into it. It was caught by chance, by an engineer who noticed his logins were half a second slow.

Agents add a new pressure. Maintainers now receive pull requests and bug reports generated by AI, some useful and many not. The curl project's maintainer has written about being flooded with AI-generated security reports that turned out to be nonsense, each one taking time to check.

## What makes a project maintainable

Most of what makes a project easy to maintain is what we have already covered:

 - **Tests and CI**, so a change that breaks something is caught by the machine (Lecture 21).
 - **A clear scope**, so the project can say no to features that do not fit.
 - **Documentation**: a README that says what it does, how to install it, and how to run the tests, plus a `CONTRIBUTING` file for anyone who wants to help.
 - **Pinned or bounded dependencies**, and a changelog.
 - **More than one person who understands it.** The *bus factor* is how many people would have to disappear before nobody could maintain the project. For most research code it is one.

If you want to give back, the easiest start is not code: a clear bug report with a small example that reproduces it, or a fix to the documentation, is genuinely useful to a maintainer.

# When to let a project die

Most projects end. The research code behind a paper, a tool built for one course, a prototype that answered its question. That is normal, and often correct: every project you keep alive costs maintenance, so keeping everything alive is not possible.

Good reasons to stop:

 - Nobody uses it any more, or it did its job (the paper is published, the decision was made).
 - Something better exists, and the effort is better spent helping that project.
 - Keeping up with dependencies costs more than the project is worth.
 - The people who understood it have moved on, and nobody else is going to learn it.

## Ending a project well

Projects rarely die cleanly. Most just stop, and the next person to find them cannot tell whether they are finished, broken or abandoned. Ending well takes very little:

 - **Say so.** Put the status at the top of the README: "No longer maintained. Use X instead." or "Archived: this is the code for paper Y, as it was published."
 - **Archive it.** GitHub can mark a repository as archived (read-only), so nobody opens issues expecting an answer.
 - **Freeze the environment**, with exact versions, so it can at least be re-run as it was.
 - **Give it a DOI** if it supports a paper. Zenodo will archive a GitHub release permanently, so the code a result came from can always be found.
 - **Hand it over** if people still depend on it, to a new maintainer or a community organisation, rather than leaving it to rot.

Python 2 is a good example of a planned death. Its end of life was announced years in advance, with a fixed date (January 1, 2020) and tools to help people move. It was still painful, but nobody was surprised.

### Question 4

The course is over, and your team's project repository is finished. Nobody plans to keep working on it. What is the most responsible thing to do?

 - A. Leave it exactly as it is
 - B. Add a status note to the README, pin the environment, make sure the tests and instructions still work, then archive it
 - C. Delete it, since nobody will maintain it
 - D. Promise in the README that it will be maintained

**B.** The project did its job, and ending it clearly is part of finishing it. Anyone who finds it later (including a future employer, or you) can see what it was, run it as it was, and knows not to expect updates. C throws away the record of what you did, and D is a promise nobody is going to keep.

# The agent builds; you decide and finish

An agent makes it cheap to write code. It does not make it cheap to maintain code, and the more you build, the more there is to keep working. So the judgement at either end of the lifecycle matters more than it did:

 - **At the start**: deciding what is worth building, for whom, and writing a spec with acceptance criteria you can test.
 - **At the end**: deciding when it is done, keeping it working for as long as people need it, and ending it cleanly when they do not.

Everything in between, the reasoning about scale, the structure, and the tests, is how you make sure what the agent builds is right.

# Recap

 - The **lifecycle**: requirements, design, implementation, testing, deployment, maintenance. **Waterfall** goes through it once; **agile** goes round it in short loops with working software each time.
 - The **V-model**: every test traces back to a level of the spec. **User stories** with **acceptance criteria** become acceptance tests, ideally written first.
 - Agents make each loop faster, which makes the **spec and the tests** more important, not less.
 - Find the **real need** before building, **prototype to think**, and choose on **speed, cost, risk and effort**.
 - **Maintenance** is most of a project's life: bugs, a changing world, and dependencies. **Deprecate** before you break.
 - Open source runs on a few people; most projects end, and **ending a project well** (status note, archive, frozen environment, DOI) is part of finishing it.

**Next (Mon Nov 23): review and a short quiz**, then the clinic, then presentations.